# Fix: nih.py v1.1 (resuelve imagenes en negro)

**Bug identificado:** `nih.py` agregaba `np.newaxis` al cargar la imagen,
y luego `NormalizeForXRV` volvia a hacer reshape. Resultado: shape (1,1,H,W)
que colapsaba la imagen al pasar por `XRayResizer`.

**Fix:** eliminar el `np.newaxis`. Dejar que `NormalizeForXRV` haga el reshape
una sola vez.

Este notebook:
1. Sobrescribe `src/datasets/nih.py` con la version corregida.
2. Vuelve a correr el smoke test completo.


## 1. Montar Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
CODE_DIR = f'{PROJECT_ROOT}/code'

import sys
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)


In [ ]:
!pip install -q torchxrayvision scipy


## 2. Sobrescribir `src/datasets/nih.py`

In [ ]:
from pathlib import Path

content = r'''"""
src/datasets/nih.py  (v1.1 - fix de normalizacion)

Dataset de PyTorch para el baseline binario NIH (No Finding vs Cardiomegaly).

FIX en v1.1:
  Se elimino el np.newaxis al cargar la imagen porque entraba en conflicto
  con reshape=True de xrv.datasets.normalize (doble reshape -> shape (1,1,H,W)
  que colapsaba la imagen a un valor constante).

  Ahora la imagen se carga como (H, W) uint8 y el primer transform
  (NormalizeForXRV) se encarga de agregar la dimension de canal.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset


class NIHDataset(Dataset):
    """
    Dataset NIH para clasificacion binaria.

    Args:
        csv_path: ruta al CSV generado por prepare_splits (train/val/test).
        images_dir: carpeta que contiene los archivos .png del NIH.
        transform: callable a aplicar sobre cada imagen (de transforms.py).

    Cada __getitem__ retorna:
        {
          'image': torch.Tensor (1, H, W), float32 en rango [-1024, 1024],
          'label': torch.Tensor escalar, long (0 o 1),
          'image_index': str (nombre del archivo, util para debugging),
        }
    """

    def __init__(self, csv_path: str | Path, images_dir: str | Path,
                 transform=None):
        self.csv_path = Path(csv_path)
        self.images_dir = Path(images_dir)
        self.transform = transform

        if not self.csv_path.exists():
            raise FileNotFoundError(f"CSV no encontrado: {self.csv_path}")
        if not self.images_dir.exists():
            raise FileNotFoundError(f"images_dir no encontrado: {self.images_dir}")

        self.df = pd.read_csv(self.csv_path)
        self._validate_columns()

    def _validate_columns(self) -> None:
        required = {'Image Index', 'label', 'Patient ID'}
        missing = required - set(self.df.columns)
        if missing:
            raise ValueError(f"Columnas faltantes en {self.csv_path}: {missing}")

    # ------------------------------------------------------------------
    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        image_name = row['Image Index']
        label = int(row['label'])

        image = self._load_image(image_name)

        if self.transform is not None:
            image = self.transform(image)

        return {
            'image': image,
            'label': torch.tensor(label, dtype=torch.long),
            'image_index': image_name,
        }

    # ------------------------------------------------------------------
    def _load_image(self, image_name: str) -> np.ndarray:
        """
        Carga una imagen como array 2D (H, W) uint8 en escala de grises.

        FIX v1.1: NO agregamos np.newaxis aqui. El primer transform
        (NormalizeForXRV, con reshape=True) se encarga de convertir
        (H, W) -> (1, H, W). Si agregaramos newaxis aqui, el reshape
        lo duplicaria y romperia el pipeline.
        """
        img_path = self.images_dir / image_name
        if not img_path.exists():
            raise FileNotFoundError(f"Imagen no encontrada: {img_path}")

        with Image.open(img_path) as pil_img:
            pil_img = pil_img.convert('L')  # forzar escala de grises
            arr = np.array(pil_img, dtype=np.uint8)

        # Devolver como (H, W). El transform agrega la dim de canal.
        return arr

    # ------------------------------------------------------------------
    def class_distribution(self) -> dict:
        counts = self.df['label'].value_counts().to_dict()
        return {
            'positive': int(counts.get(1, 0)),
            'negative': int(counts.get(0, 0)),
            'total': len(self.df),
        }


# ---------------------------------------------------------------------------
# Factory functions
# ---------------------------------------------------------------------------
def build_nih_datasets(cfg, train_transform, eval_transform
                       ) -> tuple[NIHDataset, NIHDataset, NIHDataset]:
    """
    Crea los tres datasets (train, val, test) usando la config del proyecto.
    """
    processed = Path(cfg.paths['data_processed'])
    images_dir = Path(cfg.paths['data_raw']) / cfg.dataset['images_subdir']

    train_ds = NIHDataset(
        csv_path=processed / "train.csv",
        images_dir=images_dir,
        transform=train_transform,
    )
    val_ds = NIHDataset(
        csv_path=processed / "val.csv",
        images_dir=images_dir,
        transform=eval_transform,
    )
    test_ds = NIHDataset(
        csv_path=processed / "test.csv",
        images_dir=images_dir,
        transform=eval_transform,
    )
    return train_ds, val_ds, test_ds


# ---------------------------------------------------------------------------
# Smoke test
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    import sys
    code_dir = Path(__file__).resolve().parent.parent.parent
    if str(code_dir) not in sys.path:
        sys.path.insert(0, str(code_dir))

    from src.training.utils import load_config
    from src.datasets.transforms import build_train_transform, build_eval_transform

    cfg = load_config(code_dir / "configs" / "baseline.yaml")

    train_tfm = build_train_transform(image_size=cfg.preprocessing['image_size'])
    eval_tfm = build_eval_transform(image_size=cfg.preprocessing['image_size'])

    train_ds, val_ds, test_ds = build_nih_datasets(cfg, train_tfm, eval_tfm)

    print(f"Train: {len(train_ds)} muestras | dist: {train_ds.class_distribution()}")
    print(f"Val:   {len(val_ds)} muestras | dist: {val_ds.class_distribution()}")
    print(f"Test:  {len(test_ds)} muestras | dist: {test_ds.class_distribution()}")

    sample = train_ds[0]
    print(f"\nMuestra [0]:")
    print(f"  image shape: {sample['image'].shape}")
    print(f"  image range: [{sample['image'].min():.2f}, {sample['image'].max():.2f}]")
    print(f"  label: {sample['label'].item()}")
'''
path = f'{CODE_DIR}/src/datasets/nih.py'
Path(path).write_text(content, encoding='utf-8')
print(f'OK -> {path} ({len(content):,} chars)')


## 3. Limpiar cache de imports y cargar datasets

In [ ]:
# Limpiar cache de modulos para forzar reimport
for mod in list(sys.modules.keys()):
    if mod.startswith('src.'):
        del sys.modules[mod]

from src.training.utils import load_config
from src.datasets.nih import build_nih_datasets
from src.datasets.transforms import build_train_transform, build_eval_transform

cfg = load_config(f'{CODE_DIR}/configs/baseline.yaml')

train_tfm = build_train_transform(image_size=cfg.preprocessing['image_size'])
eval_tfm = build_eval_transform(image_size=cfg.preprocessing['image_size'])

train_ds, val_ds, test_ds = build_nih_datasets(cfg, train_tfm, eval_tfm)

print(f'Train: {len(train_ds)} muestras | dist: {train_ds.class_distribution()}')
print(f'Val:   {len(val_ds)} muestras | dist: {val_ds.class_distribution()}')
print(f'Test:  {len(test_ds)} muestras | dist: {test_ds.class_distribution()}')


## 4. Visualizar una muestra (deberia verse la radiografia ahora)

In [ ]:
import matplotlib.pyplot as plt

sample = train_ds[0]
print(f"image shape: {sample['image'].shape}")
print(f"image range: [{sample['image'].min():.2f}, {sample['image'].max():.2f}]")
print(f"label: {sample['label'].item()} ({'Cardiomegaly' if sample['label'].item() == 1 else 'No Finding'})")
print(f"image_index: {sample['image_index']}")

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
img = sample['image'].squeeze().numpy()
ax.imshow(img, cmap='gray')
ax.set_title(f"{sample['image_index']} | label={sample['label'].item()}")
ax.axis('off')
plt.tight_layout()
plt.show()


## 5. Grid de 8 imagenes del batch (verificacion final)

In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
batch = next(iter(loader))

print(f"Batch shape: {batch['image'].shape}")
print(f"Rango del batch: [{batch['image'].min():.2f}, {batch['image'].max():.2f}]")
print(f"Labels positivos: {(batch['label'] == 1).sum().item()} / {len(batch['label'])}")

import matplotlib.pyplot as plt

n_show = 8
fig, axes = plt.subplots(2, 4, figsize=(14, 8))
for i, ax in enumerate(axes.flat):
    if i >= n_show:
        ax.axis('off')
        continue
    img = batch['image'][i].squeeze().numpy()
    label = batch['label'][i].item()
    ax.imshow(img, cmap='gray')
    color = 'red' if label == 1 else 'green'
    ax.set_title(f'label={label} ({"Cardio" if label else "Normal"})',
                 color=color)
    ax.axis('off')
plt.suptitle('Batch despues del fix')
plt.tight_layout()
plt.show()


## 6. Siguiente paso

Si las imagenes ahora se ven como radiografias de torax (no en negro total),
el pipeline de datos esta listo. Vuelve al chat para el modelo y las metricas.
